In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
import pandas as pd
import csv
import os
import joblib

# load the models from the file
models = joblib.load('front_view_models.joblib')

# feature and target definition
X_features = [
    'cadence_value_spm', 'gct_value_ms',
    'hip_drop_max_deg', 'shoulder_drop_max_deg',
    'foot_offset_mean_rel', 'arm_inward_mean_deg', 'fist_height_mean_rel',
    'n_ankle_x', 'n_ankle_y', 'n_knee_x', 'n_knee_y',
    'n_knee_to_hip_x', 'n_ankle_to_hip_x',
    'n_shoulder_x', 'n_elbow_x', 'n_wrist_x'
]

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

def get_pixel_point(landmarks, index, width, height):
    return np.array([int(landmarks[index].x * width), int(landmarks[index].y * height)])

# additional feature preparation
def prepare_ml_features(lm, side, frame_width, frame_height):
    if side == "Right":
        hip, knee, ankle = 24, 26, 28
        shoulder, elbow, wrist = 12, 14, 16
    else:
        hip, knee, ankle = 23, 25, 27
        shoulder, elbow, wrist = 11, 13, 15

    mid_hip_x = (lm[24].x + lm[23].x) / 2
    mid_hip_y = (lm[24].y + lm[23].y) / 2
    mid_sh_y = (lm[12].y + lm[11].y) / 2

    torso_dist = abs(mid_sh_y - mid_hip_y)
    if torso_dist == 0: torso_dist = 1

    def norm_x(idx): return (lm[idx].x - mid_hip_x) / torso_dist
    def norm_y(idx): return (lm[idx].y - mid_hip_y) / torso_dist

    features = {
        'n_ankle_x': round(norm_x(ankle), 4),
        'n_ankle_y': round(norm_y(ankle), 4),
        'n_knee_x': round(norm_x(knee), 4),
        'n_knee_y': round(norm_y(knee), 4),
        'n_knee_to_hip_x': round(norm_x(knee) - norm_x(hip), 4),
        'n_ankle_to_hip_x': round(norm_x(ankle) - norm_x(hip), 4),
        'n_shoulder_x': round(norm_x(shoulder), 4),
        'n_elbow_x': round(norm_x(elbow), 4),
        'n_wrist_x': round(norm_x(wrist), 4)
    }
    return features

# main video analysis function
def analyze_video(video_path, trained_models=None, features_list=None):
    video_capture = cv2.VideoCapture(video_path)
    fps_raw = video_capture.get(cv2.CAP_PROP_FPS)
    frames_per_second = fps_raw if fps_raw > 0 else 30.0

    # thresholds and setup
    push_off_threshold = 0.05
    strike_lockout = int(frames_per_second * 0.22)
    history_window = 9

    # state and history initialization
    right_ankle_y_history = deque(maxlen=history_window)
    left_ankle_y_history = deque(maxlen=history_window)
    cadence_history = deque(maxlen=8)
    all_step_metrics_storage = []
    
    leg_is_on_ground = {"Right": False, "Left": False}
    ankle_y_at_contact = {"Right": None, "Left": None}
    last_leg_that_landed = None
    last_strike_frame = 0
    previous_strike_time = None
    average_cadence = 0
    right_step_count, left_step_count = 0, 0
    current_status_event = "WAITING"
    latest_ai_results = {}
    last_display_frame = None

    with mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7) as pose_analyzer:
        while video_capture.isOpened():
            ret, frame = video_capture.read()
            if not ret: break

            display_frame = frame.copy()
            overlay_layer = frame.copy()
            frame_height, frame_width, _ = frame.shape
            results = pose_analyzer.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

            if results.pose_landmarks:
                landmarks = results.pose_landmarks.landmark
                
                # keypoint extraction
                r_hip = get_pixel_point(landmarks, 24, frame_width, frame_height)
                l_hip = get_pixel_point(landmarks, 23, frame_width, frame_height)
                r_shld = get_pixel_point(landmarks, 12, frame_width, frame_height)
                l_shld = get_pixel_point(landmarks, 11, frame_width, frame_height)
                r_knee = get_pixel_point(landmarks, 26, frame_width, frame_height)
                l_knee = get_pixel_point(landmarks, 25, frame_width, frame_height)
                r_ankle = get_pixel_point(landmarks, 28, frame_width, frame_height)
                l_ankle = get_pixel_point(landmarks, 27, frame_width, frame_height)
                r_elb = get_pixel_point(landmarks, 14, frame_width, frame_height)
                l_elb = get_pixel_point(landmarks, 13, frame_width, frame_height)
                r_wrst = get_pixel_point(landmarks, 16, frame_width, frame_height)
                l_wrst = get_pixel_point(landmarks, 15, frame_width, frame_height)

                # metric calculation
                mid_hip_x = (landmarks[24].x + landmarks[23].x) / 2
                hip_width = abs(landmarks[24].x - landmarks[23].x) + 1e-9
                torso_height = abs(((landmarks[23].y + landmarks[24].y)/2) - ((landmarks[11].y + landmarks[12].y)/2))

                hip_drop_deg = np.degrees(np.arctan2(abs(landmarks[24].y - landmarks[23].y), abs(landmarks[24].x - landmarks[23].x) + 1e-9))
                sh_drop_deg = np.degrees(np.arctan2(abs(landmarks[12].y - landmarks[11].y), abs(landmarks[12].x - landmarks[11].x) + 1e-9))
                
                arm_inward = (np.degrees(np.arctan2(abs(landmarks[16].x - landmarks[12].x), abs(landmarks[16].y - landmarks[12].y) + 1e-9)) + 
                              np.degrees(np.arctan2(abs(landmarks[15].x - landmarks[11].x), abs(landmarks[15].y - landmarks[11].y) + 1e-9))) / 2

                # drawing and overlay
                torso_pts = np.array([r_shld, l_shld, l_hip, r_hip], np.int32)
                cv2.fillPoly(overlay_layer, [torso_pts], (0, 255, 0)) 
                cv2.addWeighted(overlay_layer, 0.15, display_frame, 0.85, 0, display_frame)
                cv2.polylines(display_frame, [torso_pts], True, (255, 255, 255), 2)
                
                for h, k, a, color in [(r_hip, r_knee, r_ankle, (0, 255, 0)), (l_hip, l_knee, l_ankle, (0, 255, 255))]:
                    cv2.line(display_frame, tuple(h), tuple(k), color, 3)
                    cv2.line(display_frame, tuple(k), tuple(a), color, 3)
                for s, e, w in [(r_shld, r_elb, r_wrst), (l_shld, l_elb, l_wrst)]:
                    cv2.line(display_frame, tuple(s), tuple(e), (255, 165, 0), 3)
                    cv2.line(display_frame, tuple(e), tuple(w), (255, 165, 0), 3)

                current_frame_pos = int(video_capture.get(cv2.CAP_PROP_POS_FRAMES))
                right_ankle_y_history.append(landmarks[28].y)
                left_ankle_y_history.append(landmarks[27].y)

                # strike detection
                if len(right_ankle_y_history) == history_window:
                    mid_idx = history_window // 2
                    for side, history_queue, a_idx in [("Right", right_ankle_y_history, 28), ("Left", left_ankle_y_history, 27)]:
                        if (last_leg_that_landed != side and history_queue[mid_idx] == max(history_queue) and 
                            (current_frame_pos - last_strike_frame) > strike_lockout):
                            
                            leg_is_on_ground[side] = True
                            ankle_y_at_contact[side] = landmarks[a_idx].y
                            last_leg_that_landed = side
                            last_strike_frame = current_frame_pos
                            
                            if previous_strike_time is not None:
                                cadence_history.append((60 * frames_per_second) / (current_frame_pos - previous_strike_time))
                                average_cadence = np.mean(cadence_history)
                            previous_strike_time = current_frame_pos
                            
                            if side == "Right": right_step_count += 1
                            else: left_step_count += 1
                            current_status_event = f"{side.upper()} STRIKE"
                            
                            ml_data = prepare_ml_features(landmarks, side, frame_width, frame_height)
                            all_step_metrics_storage.append({
                                'side': side, 'start_frame': current_frame_pos, 'done': False,
                                'cadence_value_spm': average_cadence,
                                'hip_drop_max_deg': hip_drop_deg,
                                'shoulder_drop_max_deg': sh_drop_deg,
                                'arm_inward_sum': arm_inward,
                                'fist_height_sum': (landmarks[0].y - landmarks[16 if side=="Right" else 15].y) / (torso_height + 1e-9),
                                'foot_offset_sum': (landmarks[a_idx].x - mid_hip_x) / hip_width,
                                'n': 1, **ml_data
                            })

                # push-off detection and ml prediction
                for side, a_idx in [("Right", 28), ("Left", 27)]:
                    if leg_is_on_ground[side]:
                        for entry in reversed(all_step_metrics_storage):
                            if entry['side'] == side and not entry['done']:
                               
                                entry['hip_drop_max_deg'] = max(entry['hip_drop_max_deg'], hip_drop_deg)
                                entry['shoulder_drop_max_deg'] = max(entry['shoulder_drop_max_deg'], sh_drop_deg)
                                entry['arm_inward_sum'] += arm_inward
                                entry['fist_height_sum'] += (landmarks[0].y - landmarks[16 if side=="Right" else 15].y) / (torso_height + 1e-9)
                                entry['foot_offset_sum'] += (landmarks[a_idx].x - mid_hip_x) / hip_width
                                entry['n'] += 1
                                
                                frames_on_ground = current_frame_pos - entry['start_frame']
                                if (ankle_y_at_contact[side] - landmarks[a_idx].y) > push_off_threshold and frames_on_ground >= 5:
                                    entry['gct_value_ms'] = (frames_on_ground / frames_per_second) * 1000
                                    entry['arm_inward_mean_deg'] = entry['arm_inward_sum'] / entry['n']
                                    entry['fist_height_mean_rel'] = entry['fist_height_sum'] / entry['n']
                                    entry['foot_offset_mean_rel'] = entry['foot_offset_sum'] / entry['n']
                                    entry['done'] = True
                                    leg_is_on_ground[side] = False
                                    current_status_event = f"{side.upper()} PUSH-OFF"
                                    
                                    if trained_models:
                                        step_df = pd.DataFrame([entry])
                                        for target, model in trained_models.items():
                                            latest_ai_results[target] = model.predict(step_df[features_list])[0]
                                break

                # visualizations
                cv2.rectangle(display_frame, (0, 0), (280, 100), (20, 20, 20), -1)
                cv2.putText(display_frame, f"STEPS: {right_step_count + left_step_count}", (15, 35), 1, 1.8, (255, 255, 255), 2)
                cv2.putText(display_frame, f"CADENCE: {int(average_cadence)}", (15, 65), 1, 1.2, (0, 255, 0), 2)
                cv2.putText(display_frame, f"STATUS: {current_status_event}", (15, 95), 1, 1.0, (0, 255, 255), 1)

                if latest_ai_results:
                    bw, start_y, sx = 250, 35, frame_width - 260
                    cv2.rectangle(display_frame, (sx, 10), (frame_width - 10, start_y + 160), (20, 20, 20), -1)
                    cv2.putText(display_frame, "AI ANALYSIS", (sx + 15, start_y), 1, 1.2, (255, 255, 255), 2)
                    for i, (metric, score) in enumerate(latest_ai_results.items()):
                        y_p = start_y + 30 + (i * 22)
                        val = int(score)
                        color = (0, 255, 0) if val >= 3 else (0, 255, 255) if val == 2 else (0, 0, 255)
                        cv2.putText(display_frame, f"{metric.replace('_score','').upper()}:", (sx + 15, y_p), 1, 0.8, (200, 200, 200), 1)
                        cv2.putText(display_frame, f"{val}", (sx + bw - 30, y_p), 1, 1.0, color, 2)

                for side_ui, pos, state in [("LEFT", (10, frame_height-20), leg_is_on_ground["Left"]), ("RIGHT", (frame_width-130, frame_height-20), leg_is_on_ground["Right"])]:
                    color = (0, 255, 0) if state else (0, 0, 255)
                    cv2.rectangle(display_frame, (pos[0]-10, pos[1]-40), (pos[0]+120, pos[1]+10), (0,0,0), -1)
                    cv2.putText(display_frame, side_ui, (pos[0], pos[1]-20), 1, 1.2, color, 2)
                    cv2.putText(display_frame, "CONTACT" if state else "FLIGHT", (pos[0], pos[1]), 1, 0.9, (255,255,255), 1)

                cv2.imshow('Running Analysis - Front View', display_frame)
                last_display_frame = display_frame.copy()
                if cv2.waitKey(1) & 0xFF == ord('q'): break

    # session summary screen
    if last_display_frame is not None and trained_models:
        test_df = pd.DataFrame(all_step_metrics_storage)
        test_df = test_df[test_df['done'] == True].copy()
        if not test_df.empty:
            overlay = last_display_frame.copy()
            cv2.rectangle(overlay, (0, 0), (frame_width, frame_height), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.7, last_display_frame, 0.3, 0, last_display_frame)
            cx, cy = frame_width // 2, frame_height // 2
            cv2.rectangle(last_display_frame, (cx - 250, cy - 225), (cx + 250, cy + 225), (30, 30, 30), -1)
            cv2.putText(last_display_frame, "SESSION SUMMARY", (cx - 180, cy - 180), 1, 2.0, (255, 255, 255), 2)

            for i, (target, model) in enumerate(trained_models.items()):
                avg_score = int(round(np.mean(model.predict(test_df[features_list]))))
                color = (0, 255, 0) if avg_score >= 3 else (0, 255, 255) if avg_score == 2 else (0, 0, 255)
                y_p = cy - 100 + (i * 35)
                cv2.putText(last_display_frame, f"{target.replace('_score','').upper()}:", (cx - 220, y_p), 1, 1.2, (200, 200, 200), 1)
                cv2.putText(last_display_frame, f"{avg_score}", (cx + 180, y_p), 1, 1.5, color, 2)

            cv2.imshow('Running Analysis - Front View', last_display_frame)
            cv2.waitKey(0)

    video_capture.release()
    cv2.destroyAllWindows()
    return all_step_metrics_storage

# execution
video_name = './Videos/tr1.mp4'
results = analyze_video(video_name, models, X_features)

I0000 00:00:1769742730.157277 1249209 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1769742730.230884 1307994 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769742730.240686 1307999 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769742730.253884 1307993 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
